# 12 - Matriz de experimentos con los hiperparámetros ajustados
En este notebook comparamos la matriz completa de experimentos del notebook 10 con la misma matriz usando los hiperparámetros ajustados en el notebook 11.

Las dos matrices se han ejecutado con `make experiments` (`urban_accessibility_ml/modeling/experiments_run.py`), con las mismas particiones, conjuntos de variables, perfiles y modelos: una con los valores por defecto de cada modelo y otra con los hiperparámetros ajustados de cada partición. Tardan varios minutos, así que aquí solo se leen los resultados, que están en `reports/results/`:
- `experiment-matrix-default.csv`: con los valores por defecto.
- `experiment-matrix-tuned.csv`: con los hiperparámetros ajustados.

El ajuste solo ha usado los datos de entrenamiento de cada partición, así que el test sigue sin haberse usado para elegir nada. Aquí vemos cuánto mejoran los modelos en el test.

## Carga de los resultados

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from urban_accessibility_ml.config import REPORTS_DIR
from urban_accessibility_ml.modeling.evaluate import summarize_folds
from urban_accessibility_ml.plotting.palettes import SERIE_BASE
from urban_accessibility_ml.plotting.plots import SERIES, use_style

DEFAULT_FILE = REPORTS_DIR / "results" / "experiment-matrix-default.csv"
TUNED_FILE = REPORTS_DIR / "results" / "experiment-matrix-tuned.csv"
GROUP = ["strategy", "model"]
STRATEGIES = ["within_city", "mixed", "cross_city"]

use_style()

results_default = pd.read_csv(DEFAULT_FILE)
results_tuned = pd.read_csv(TUNED_FILE)
len(results_default), len(results_tuned)

## Comparación por estrategia y modelo
F1 macro media en el test, con los valores por defecto y con los ajustados. Cada fila contiene los promedios de todos los conjuntos de variables, perfiles y pliegues de esa estrategia y modelo. La última columna es la diferencia: si es positiva, el ajuste mejora el resultado.

In [ ]:
default = summarize_folds(results_default, GROUP).set_index(GROUP)["f1_macro_mean"]
tuned = summarize_folds(results_tuned, GROUP).set_index(GROUP)["f1_macro_mean"]

comparison = pd.DataFrame({"por defecto": default, "ajustado": tuned})
comparison["diferencia"] = comparison["ajustado"] - comparison["por defecto"]
comparison.round(3)

In [ ]:
fig, axes = plt.subplots(1, len(STRATEGIES), figsize=(13, 4), sharey=True)
width = 0.38

for ax, strategy in zip(axes, STRATEGIES):
    part = comparison.loc[strategy]
    x = np.arange(len(part))
    ax.bar(x - width / 2, part["por defecto"], width, color=SERIE_BASE, label="por defecto")
    ax.bar(x + width / 2, part["ajustado"], width, color=SERIES[0], label="ajustado")
    ax.set_xticks(x, part.index, rotation=30, ha="right")
    ax.set_title(strategy)

axes[0].set_ylabel("F1 macro (test)")
axes[0].legend()
plt.tight_layout()
plt.show()

Vemos que en la regresión logística sí se aprecian mejoras, sobre todo en misma ciudad (pasa de 0,754 a 0,838) y en mixto (de 0,747 a 0,809). En cambio, en cross_city casi no cambia.

En los modelos de árboles la historia es distinta: no se aprecian mejoras y, en algunos casos, incluso empeoran un poco. Las diferencias son pequeñas, y la mayor caída es la de XGBoost en misma ciudad (de 0,888 a 0,867).